In [81]:
import os
import pandas as pd
import numpy as np
from pydub import AudioSegment
from tqdm import tqdm

In [68]:
def safe_mkdir(path):
    if not os.path.exists(path):
        os.mkdir(path)

In [57]:
def cut_audio(infile, start, end):
    audio = AudioSegment.from_file(infile)
    seg = audio[int(start*1000):int(end*1000)]
    seg.export(outfile, format="mp3")
    return seg

In [5]:
file_list = pd.read_csv('../DementiaBank-preprocessed2/file_list.csv')
file_list

,Unnamed: 0.1,Unnamed: 0,path,site,group,filename,diag,subName,size_MB,dur_s,dur_m
0,0,0,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,033-1.mp3,D,Pitt_D_033,2.38,155.83,2.597167
1,1,1,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,033-0.mp3,D,Pitt_D_033,0.85,55.58,0.926333
2,2,2,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,076-0.mp3,D,Pitt_D_076,1.25,81.83,1.363833
3,3,3,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,049-0.mp3,D,Pitt_D_049,1.07,70.18,1.169667
4,4,4,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Pitt,dementia,076-2.mp3,D,Pitt_D_076,1.45,95.13,1.585500
...,...,...,...,...,...,...,...,...,...,...,...
280,280,56,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,33-1.mp3,D,Delaware_D_33,6.50,425.59,7.093167
281,281,57,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,15-2.mp3,D,Delaware_D_15,8.86,857.43,14.290500
282,282,58,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,69-1.mp3,D,Delaware_D_69,4.25,278.69,4.644833
283,283,59,/Users/aidasaglinskas/Desktop/ANDSpeak/Google_...,Delaware,dementia,11-1.mp3,D,Delaware_D_11,15.24,998.52,16.642000


In [58]:
indir = '../DementiaBank-preprocessed2/02-patient-speaker-files/patient-dfs/'
files = [f for f in os.listdir(indir) if f.endswith('.csv')]
files.sort()

In [82]:
for f in tqdm(range(len(files))):
    file = files[f]
    df = pd.read_csv(os.path.join(indir,file))
    idx = df['duration_sec'].values>2.0
    df = df.iloc[idx]
    
    if len(df)>0:
        j = file_list['filename'].values==file.replace('.csv','.mp3')
        audio_path = file_list.iloc[j]['path'].values[0]
        
        for i in range(df.shape[0]):
            start = df['start_sec'].values[i]
            end = df['end_sec'].values[i]
            
            audio = AudioSegment.from_file(audio_path)
            buffer = 500
            seg = audio[int(start*1000)-buffer:int(end*1000)+buffer]
            
            outdir = '../DementiaBank-preprocessed2/02-patient-speaker-files/patient-audio/'
            safe_mkdir(os.path.join(outdir,file.replace('.csv','')))
            ofn = os.path.join(outdir,file.replace('.csv',''),file.replace('.csv',f'_{i}.mp3'))
            seg.export(ofn, format="mp3");

100%|███████████████████████████████████████████████████████████████████████████████████████████| 258/258 [8:16:33<00:00, 115.48s/it]
